# 02 – Data Exploration
Download and inspect OHLCV data via yfinance (no broker needed).  
Useful for backtesting and understanding market dynamics.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from core.data_fetcher import get_data

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
# ── Choose your tickers ─────────────────────────────────────────────────────
# Irish/European stocks: use Yahoo Finance symbols
# e.g. AIB.IR (AIB Group), CRH.L (CRH on LSE), AAPL (US)
TICKERS = ['CRH.L', 'SHEL.L', 'AAPL', 'MSFT']

data = {t: get_data(t, source='yfinance', period='2y') for t in TICKERS}
for t, df in data.items():
    print(f'{t:12s}  rows={len(df)}  from={df.index[0].date()}  to={df.index[-1].date()}')

In [ ]:
# Normalised price chart (rebased to 100)
fig, ax = plt.subplots()
for t, df in data.items():
    (df['close'] / df['close'].iloc[0] * 100).plot(ax=ax, label=t)
ax.legend()
ax.set_title('Normalised price (base = 100)')
ax.set_ylabel('Index')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rolling 20-day volatility
fig, ax = plt.subplots()
for t, df in data.items():
    vol = df['close'].pct_change().rolling(20).std() * (252**0.5) * 100
    vol.plot(ax=ax, label=t)
ax.set_title('Annualised 20-day rolling volatility (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of daily returns
import seaborn as sns
returns = pd.DataFrame({t: data[t]['close'].pct_change() for t in TICKERS}).dropna()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(returns.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Return correlation matrix')
plt.tight_layout()
plt.show()